<a href="https://colab.research.google.com/github/Ena-AlexBrush/Fine-Tuning-Experiments/blob/main/GRPO_ChatTemplate_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [2]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 46.8 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
!pip install trl[GRPOTrainer]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [4]:
from datasets import load_dataset
from trl import GRPOTrainer, GRPOConfig
from trl.rewards import accuracy_reward
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import re

In [22]:
# Dataset Prep
t_dataset = load_dataset("HuggingFaceTB/smoltalk", "smol-magpie-ultra", split="train[:20]")
e_dataset = load_dataset("HuggingFaceTB/smoltalk", "smol-magpie-ultra", split="test[:20]")

# train_dataset = t_dataset.rename_column("messages", "prompt")
# eval_dataset = e_dataset.rename_column("messages", "prompt")

In [23]:
# def generate_r1_prompt(tokenizer):
#     messages = [
#         {"role": "user", "content": "Hello, how are you?"},
#         {"role": "assistant", "content": "I'm doing great. How can I help you today?"},
#         {"role": "user", "content": "I'd like to show off how chat templating works!"},
#     ]
#     return {
#         "prompt": tokenizer.apply_chat_template(messages, tokenizer=False, add_generation_prompt=True)
#     }


# chat = [
#   {"role": "user", "content": "Hello, how are you?"},
#   {"role": "assistant", "content": "I'm doing great. How can I help you today?"},
#   {"role": "user", "content": "I'd like to show off how chat templating works!"},
# ]

def extract_prompt(example):
    messages = example["messages"]
    user_only = [m for m in messages if m["role"] == "user"]
    return {"prompt": [user_only[-1]]}

train_dataset = t_dataset.map(extract_prompt, remove_columns=t_dataset.column_names)
eval_dataset = e_dataset.map(extract_prompt, remove_columns=e_dataset.column_names)

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM-135M", device_map="auto")

tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<|im_start|>assistant\n' }}"
    "{% endif %}"
)

# tokenizer.apply_chat_template(chat, tokenize=False)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [ ]:
# # Not really sure what the point of this is... since i already have a funciton
# train_dataset = train_dataset.map(lambda x: generate_r1_prompt(x))

In [30]:
def format_reward_func(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0.0
        # completion is a list of dicts with 'role' and 'content'
        text = completion[0]["content"].strip()
        if text and text[-1] in ".!?":
            score += 1.0
        if 20 <= len(text) <= 200:
            score += 0.5
        scores.append(score)
    return scores

In [25]:
training_args = GRPOConfig(
    output_dir="output",
    num_train_epochs=3,
    num_generations=4,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-5,
    logging_steps=10,
    max_completion_length=128,
)


In [31]:
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    processing_class=tokenizer,
    reward_funcs=format_reward_func,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)


In [ ]:
# print(train_dataset[0]["prompt"])

In [21]:
print(type(train_dataset[0]["prompt"]))
print(train_dataset[0]["prompt"])
print(tokenizer.chat_template)

<class 'list'>
[{'content': 'Jim has won a vacation at a lake resort and can choose to travel to either lake A or lake B. Lake A has a 20% chance of rain spoiling the vacation while lake B has a 40% chance of rain. The cost of travel to lake A is $200 higher than that to lake B. What are some factors Jim might consider in making a decision on which lake to choose.', 'role': 'user'}, {'content': "To make an informed decision, Jim should weigh the potential risks and costs associated with each option. One key factor to consider is the likelihood of rain spoiling the vacation. Lake A has a lower chance of rain, at 20%, which increases the likelihood of a pleasant vacation. On the other hand, lake B has a higher chance of rain, at 40%, which may lead to a less enjoyable experience.\n\nAnother crucial factor is the cost difference between traveling to lake A and lake B. With a $200 higher cost for lake A, Jim needs to consider whether the lower risk of rain is worth the additional expense. 

In [32]:
trainer.train()

Step,Training Loss
10,0.047728
20,0.087969
30,0.060699


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=30, training_loss=0.065465181072553, metrics={'train_runtime': 260.1136, 'train_samples_per_second': 0.231, 'train_steps_per_second': 0.115, 'total_flos': 0.0, 'train_loss': 0.065465181072553, 'epoch': 3.0})

In [33]:
trainer.evaluate()

Training Loss,Validation Loss,Step,Num Tokens,Completions/mean Length,Completions/min Length,Completions/max Length,Completions/clipped Ratio,Completions/mean Terminated Length,Completions/min Terminated Length,Completions/max Terminated Length,Rewards/format Reward Func/mean,Rewards/format Reward Func/std,Reward,Reward Std,Frac Reward Zero Std,Entropy,Clip Ratio/low Mean,Clip Ratio/high Mean,Clip Ratio/region Mean,Clip Ratio/low Min,Clip Ratio/high Max
0.060699,0.055339,30,46363.000000,119.862500,78.900000,128.000000,0.875000,39.916667,27.700000,53.500000,0.062500,0.132903,0.062500,0.132903,0.750000,3.923982,0.000000,0.000000,0.000000,0.000000,0.000000


{'eval_loss': 0.05533861368894577,
 'eval_num_tokens': 46363.0,
 'eval_completions/mean_length': 119.8625,
 'eval_completions/min_length': 78.9,
 'eval_completions/max_length': 128.0,
 'eval_completions/clipped_ratio': 0.875,
 'eval_completions/mean_terminated_length': 39.91666679382324,
 'eval_completions/min_terminated_length': 27.7,
 'eval_completions/max_terminated_length': 53.5,
 'eval_rewards/format_reward_func/mean': 0.0625,
 'eval_rewards/format_reward_func/std': 0.13290322721004486,
 'eval_reward': 0.0625,
 'eval_reward_std': 0.13290322721004486,
 'eval_frac_reward_zero_std': 0.75,
 'eval_entropy': 3.9239821910858153,
 'eval_clip_ratio/low_mean': 0.0,
 'eval_clip_ratio/high_mean': 0.0,
 'eval_clip_ratio/region_mean': 0.0,
 'eval_clip_ratio/low_min': 0.0,
 'eval_clip_ratio/high_max': 0.0}